In [1]:
from pathlib import Path 
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MKL_SERVICE_FORCE_INTEL'] = '1'
from tqdm import tqdm
from IPython.display import Video

import torch
import numpy as np

import sys
sys.path.append("/srv/sferraro/choreographer/")

import envs
from envs.make import make

import matplotlib.pyplot as plt
import matplotlib.animation as animation

import mani_skill2
import mani_skill2.envs

import utils
%matplotlib inline
from IPython import display
import imageio


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/elephant/miniconda3/envs/urlb/lib/python3.8/site-packages/robosuite-1.4.0-py3.8.egg/robosuite/scripts/setup_macros.py (__init__.py:9)


In [2]:
from hydra import compose, initialize
from omegaconf import OmegaConf

initialize(config_path="../../exp_local/2023.05.23/131105_dreamer_obj_rsPanda_CustomLift_dense_rw/.hydra", job_name="config")
cfg = compose(config_name="config")

In [3]:
# import os
# main_model_dir = "/srv/sferraro/choreographer/notebooks/paper_visualizations/models/"
# envs_name = [x[1] for x in os.walk(main_model_dir)][0]

envs_dict = {"cube": "rsPanda_CustomLift", "stack": "rsPanda_CustomStack"}
snapshot_size = [100000, 500000, 1000000, 2000000]

def load_agent(agent_path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    with agent_path.open('rb') as f:
        obj = torch.load(f, map_location=torch.device(device))
        agent = obj['agent']    
        step = obj['_global_step']
        agent.device = device
        agent.wm.device = device
        agent.wm.rssm.device = device
        agent.wm.rssm._cell.device = device
    return agent, step


In [20]:
# WM comparison gifs
for model_dir, env_name in envs_dict.items():
    
    # Import agents models (WM + Actor Critic)
    agent_path = Path(f'/srv/sferraro/choreographer/notebooks/paper_visualizations/models/{model_dir}/dreamer_obj/snapshot_2000000.pt')
    dreamer_agent_path = Path(f'/srv/sferraro/choreographer/notebooks/paper_visualizations/models/{model_dir}/dreamer/snapshot_2000000.pt')

    FOCUS_agent, global_step = load_agent(agent_path)
    dreamer_agent, global_step = load_agent(dreamer_agent_path)

    #config variables
    obs_type = dreamer_agent.cfg.obs_type
    action_repeat = dreamer_agent.cfg.action_repeat
    snapshot_ts = global_step * action_repeat

    dreamer_agent.reward_free = True
    FOCUS_agent.reward_free = True
    dreamer_agent.use_selector = False
    FOCUS_agent.use_selector = False
    dreamer_agent.detached_exploration = True
    FOCUS_agent.detached_exploration = True

    seed = dreamer_agent.cfg.seed

    task = env_name
    domain = task.split("_")[0]

    cfg.env.objects.minsize = 0.025
    cfg.env.controller = "OSC_POSE" if domain == "rsPanda" else "pd_ee_delta_pos"
    camera = cfg.env.renderer.camera = "agentview2" if domain == "rsPanda" else "base_camera"

    cfg.env.objects.spawn_range = 0.001
    cfg.env.objects.random_placement = False
    # Env creation
    env = make(task, obs_type, frame_stack=1, 
                        action_repeat=action_repeat, seed=seed, env_config=cfg.env)
    
    newpath = r'./temp' 
    if not os.path.exists(newpath):
        os.makedirs(newpath)

    render_size = 64

    eval_mode = True

    agent_state = None
    dreamer_meta = dreamer_agent.init_meta()
    FOCUS_meta = FOCUS_agent.init_meta()

    obs = env.reset()
    episodes = 3

    step, episode, total_reward = 0, 0, 0

    instances = 2 if task == "rsPanda_CustomLift" else 3

    for ep in tqdm(range(episodes)):
        dreamer_state, FOCUS_state = None, None
        obs = env.reset()
        step, total_reward = 0, 0
        
        frames = []
                
        with torch.no_grad(), utils.eval_mode(dreamer_agent), utils.eval_mode(FOCUS_agent):
            while not bool(obs['is_last']):

                _, dreamer_state = dreamer_agent.act(
                                        obs,
                                        dreamer_meta,
                                        step,
                                        eval_mode=False,
                                        state=dreamer_state,
                                    )
                
                _, FOCUS_state = FOCUS_agent.act(
                                        obs,
                                        FOCUS_meta,
                                        step,
                                        eval_mode=False,
                                        state=FOCUS_state,
                                    )
                
                action = np.random.uniform(-1, 1, size=(dreamer_agent.act_dim)).astype(np.float32)
                
                dreamer_state = (dreamer_state[0], torch.tensor(action).to(dreamer_agent.device).unsqueeze(0)) 
                FOCUS_state = (FOCUS_state[0], torch.tensor(action).to(dreamer_agent.device).unsqueeze(0)) 
                
                obs = env.step(action)
                
                d_feat = dreamer_agent.wm.rssm.get_feat(dreamer_state[0]).unsqueeze(0)
                f_feat = FOCUS_agent.wm.rssm.get_feat(FOCUS_state[0]).unsqueeze(0)

                seg = torch.tensor((obs["segmentation"].copy())).unsqueeze(0).unsqueeze(0).to(FOCUS_agent.device)
                rgb = torch.tensor((obs["rgb"].copy())).to(FOCUS_agent.device) / 255.0
                
                f_prediction = FOCUS_agent.wm.heads["object_decoder"](f_feat, seg * 1.0)
                d_prediction = dreamer_agent.wm.heads["decoder"](d_feat)
                
                f_obs = f_prediction["rgb"].mean[0,0] + 0.5

                d_obs = d_prediction["rgb"].mean[0,0] + 0.5
                
                predicted_segmap = f_prediction["segmentation"].mean[0,0]
                
                # asssemble full predicted image
                f_out = torch.zeros((render_size, render_size, 3), device=dreamer_agent.device)
                for instance in range(instances):
                    pred_seg_channel = predicted_segmap.permute(2, 0, 1)[instance].unsqueeze(-1)
                    f_out = f_out + f_obs[instance].permute(1, 2, 0) * pred_seg_channel.repeat(1,1,3)
                    
                out = torch.cat((rgb, f_out.permute(2,0,1), d_obs), 1)

                step += 1
                instances = 2 if task == "rsPanda_CustomLift" else 3
                frames.append(out.permute(1,2,0).cpu().numpy())
            
                # ax.imshow(out.permute(1,2,0).cpu().numpy())
                # plt.savefig(f'./images/{task}_{episode}.png', transparent = False, facecolor="white", bbox_inches='tight')
        
        episode += 1
        
        # for t in range(step):
        #     image = imageio.v2.imread(f'./temp/img_{t}_{episode}.png')
        #     frames.append(image)
            
        imageio.mimsave(f'./out_{env_name}_{episode}.gif', # output gif
                    frames,          # array of input frames
                    fps = 10)  

100%|██████████| 3/3 [00:14<00:00,  4.75s/it]


In [7]:
from copy import deepcopy

# Exploration gifs
for model_dir, env_name in envs_dict.items():
    for snapshot in snapshot_size:
        # Import agents models (WM + Actor Critic)
        agent_path = Path(f'/srv/sferraro/choreographer/notebooks/paper_visualizations/models/{model_dir}/dreamer_obj/snapshot_{snapshot}.pt')

        FOCUS_agent, global_step = load_agent(agent_path)

        #config variables
        obs_type = FOCUS_agent.cfg.obs_type
        action_repeat = FOCUS_agent.cfg.action_repeat
        snapshot_ts = global_step * action_repeat

        FOCUS_agent.reward_free = True
        FOCUS_agent.use_selector = False
        FOCUS_agent.detached_exploration = True

        seed = FOCUS_agent.cfg.seed

        task = env_name
        domain = task.split("_")[0]

        cfg.env.objects.minsize = 0.025
        cfg.env.controller = "OSC_POSE" if domain == "rsPanda" else "pd_ee_delta_pos"
        camera = cfg.env.renderer.camera = "agentview2" if domain == "rsPanda" else "base_camera"

        cfg.env.objects.spawn_range = 0.001
        cfg.env.objects.random_placement = False
        
        #Env creation
        env = make(task, obs_type, frame_stack=1, 
                            action_repeat=action_repeat, seed=seed, env_config=cfg.env)

        newpath = f'./gifs/{model_dir}' 
        if not os.path.exists(newpath):
            os.makedirs(newpath)

        render_size = 64

        eval_mode = True

        agent_state = None
        FOCUS_meta = FOCUS_agent.init_meta()

        obs = env.reset()
        episodes = 3

        step, episode, total_reward = 0, 0, 0

        instances = 2

        for ep in tqdm(range(episodes)):
            FOCUS_state = None
            obs = env.reset()
            step, total_reward = 0, 0
            
            frames = []
            actions = []
                    
            with torch.no_grad(), utils.eval_mode(FOCUS_agent):
                while not bool(obs['is_last']):
                    
                    action, FOCUS_state = FOCUS_agent.act(
                                            obs,
                                            FOCUS_meta,
                                            step,
                                            eval_mode=False,
                                            state=FOCUS_state,
                                        )
                    
                    actions.append(action)
                    
                    obs = env.step(action).copy()
                    
                    # f_feat = FOCUS_agent.wm.rssm.get_feat(FOCUS_state[0]).unsqueeze(0)

                    # seg = torch.tensor((obs["segmentation"].copy())).unsqueeze(0).unsqueeze(0).to(FOCUS_agent.device)
                    # rgb = torch.tensor((obs["rgb"].copy())).to(FOCUS_agent.device) / 255.0
                    # rgb_render = obs_render["rgb"].copy()
                    
                    # f_prediction = FOCUS_agent.wm.heads["object_decoder"](f_feat, seg * 1.0)
                    
                    # f_obs = f_prediction["rgb"].mean[0,0] + 0.5
                    
                    # predicted_segmap = f_prediction["segmentation"].mean[0,0]
                    
                    # asssemble full predicted image
                    # f_out = torch.zeros((render_size, render_size, 3), device=dreamer_agent.device)
                    # for instance in range(instances):
                    #     pred_seg_channel = predicted_segmap.permute(2, 0, 1)[instance].unsqueeze(-1)
                    #     f_out = f_out + f_obs[instance].permute(1, 2, 0) * pred_seg_channel.repeat(1,1,3)
                        
                    # out = torch.clip(torch.cat((rgb, f_out.permute(2,0,1)), 1), 0, 1) 

                    step += 1
                    
                    # frames.append(rgb_render.transpose(1,2,0))
        
            cfg_render = deepcopy(cfg)
            cfg_render.env.renderer.size = [128, 128]
            
            env_render= make(task, obs_type, frame_stack=1, 
                                action_repeat=action_repeat, seed=seed, env_config=cfg_render.env)
            env_render.reset()
            
            for action in actions:
                obs_render = env_render.step(action).copy()
                frames.append(obs_render["rgb"].transpose(1,2,0))          
            
            episode += 1
                
            imageio.mimsave(f'./gifs/{model_dir}/expl_{env_name}_{snapshot}_{episode}.gif', # output gif
                        frames,          # array of input frames
                        fps = 10)  


100%|██████████| 3/3 [00:21<00:00,  7.24s/it]
